The JSON will be structured in a way so that the LLM understands:

1. The Intent: What is the user actually looking for?

2. The Logic: The exact SQL WHERE clauses or Python math you used.

3. The Tables: Which parts of your 5-table schema to join.

### Dictionary Making:

In [ ]:
# First 19 Red Flags are included:

import json
import os

# Define the absolute path to your project folder
project_root = r"B:\3. Prog\2. Projects\7. Logistics and supply chain"
metadata_folder = os.path.join(project_root, "metadata")

# Ensure the folder exists
os.makedirs(metadata_folder, exist_ok=True)

# THE 19 RED FLAGS (Extracted from Logistics_EDA_Automation.ipynb)
red_flags_logic = [
    {"name": "RF01: Thermal Compliance Crisis", "description": "Global count of shipments where iot_temperature is < 0 or > 15 (11,598 breaches).", "sql": "iot_temperature NOT BETWEEN 0 AND 15", "tables": ["fact_shipments"]},
    {"name": "RF02: Supplier Integrity Audit", "description": "Identification of carriers with frequent temperature violations above 8.0C.", "sql": "iot_temperature > 8.0 GROUP BY supplier_id", "tables": ["fact_shipments", "dim_suppliers"]},
    {"name": "RF03: Status Casting Resolution", "description": "Logic to handle order_fulfillment_status as VARCHAR to prevent numeric conversion errors.", "sql": "CAST(order_fulfillment_status AS VARCHAR)", "tables": ["fact_shipments"]},
    {"name": "RF04: Thermal Volatility (RTE_01007)", "description": "Statistical variance audit to find inconsistent cooling on specific routes.", "sql": "route_id = 'RTE_01007' AND STDDEV(iot_temperature)", "tables": ["fact_shipments", "dim_routes"]},
    {"name": "RF05: Bimodal Peaks (RTE_00727)", "description": "Detection of two distinct temperature clusters indicating cooling equipment failure.", "sql": "route_id = 'RTE_00727'", "tables": ["fact_shipments"]},
    {"name": "RF06: Delay Frequency Ceiling", "description": "Top 10 routes where delivery_time_deviation > 0 occurs in over 96% of cases.", "sql": "AVG(CASE WHEN delivery_time_deviation > 0 THEN 1 ELSE 0 END) > 0.96", "tables": ["fact_shipments", "dim_routes"]},
    {"name": "RF07: Supplier SUP_00726 Failure", "description": "Documenting specific supplier with a near-total (>99%) delay rate.", "sql": "supplier_id = 'SUP_00726'", "tables": ["fact_shipments"]},
    {"name": "RF08: Supplier SUP_00548 Failure", "description": "Documenting secondary supplier with a near-total (>99%) delay rate.", "sql": "supplier_id = 'SUP_00548'", "tables": ["fact_shipments"]},
    {"name": "RF09: ID Padding Resolution", "description": "Handling leading zero inconsistencies for supplier IDs (e.g. SUP_00875 vs SUP_875).", "sql": "supplier_id LIKE '%875%'", "tables": ["fact_shipments"]},
    {"name": "RF10: 39.25% FTR Ceiling", "description": "Measuring the cap on 'Perfect Shipments' (On-Time + Thermal Integrity) for SUP_00358.", "sql": "supplier_id = 'SUP_00358'", "tables": ["fact_shipments"]},
    {"name": "RF11: 47.64% Cost Variance", "description": "Isolating supplier billing anomalies compared to the global network average.", "sql": "supplier_id = 'SUP_00484' AND AVG(shipping_costs)", "tables": ["fact_shipments"]},
    {"name": "RF12: Statistical Outlier (Z-Score)", "description": "Isolating extreme tail-risk incidents where temperature Z-Score exceeds 4.5.", "sql": "(iot_temperature - avg) / stddev > 4.5", "tables": ["fact_shipments"]},
    {"name": "RF13: $28M Spend Concentration", "description": "Single-point-of-failure risk based on total financial volume for SUP_00101.", "sql": "SUM(shipping_costs) BY supplier_id", "tables": ["fact_shipments"]},
    {"name": "RF14: Bottleneck RTE_00052", "description": "Quantifying total time-loss impact (Volume * Avg Delay) for the most congested route.", "sql": "route_id = 'RTE_00052'", "tables": ["fact_shipments"]},
    {"name": "RF15: Mean Imputation Artifact", "description": "Detecting data quality flatlining where 18.6C is over-represented in sensor data.", "sql": "iot_temperature = 18.6", "tables": ["fact_shipments"]},
    {"name": "RF16: Geospatial Hot-Zones", "description": "Mapping failure clusters using 11.1km GPS coordinate rounding (1 decimal place).", "sql": "ROUND(lat, 1), ROUND(lon, 1)", "tables": ["fact_shipments", "dim_routes"]},
    {"name": "RF17: Cost Volatility Audit", "description": "Measuring standard deviation in billing to identify anomalous supplier behavior.", "sql": "STDDEV(shipping_costs) BY supplier_id", "tables": ["fact_shipments"]},
    {"name": "RF18: Temperature-Linked Delays", "description": "Correlation check between thermal compliance breaches and delivery time deviation.", "sql": "iot_temperature > 15 AND delivery_time_deviation > 0", "tables": ["fact_shipments"]},
    {"name": "RF19: 0.1% Catastrophic Deviation", "description": "Isolating the worst-performing 0.1% of delivery time outliers for risk mitigation.", "sql": "PERCENT_RANK(delivery_time_deviation) > 0.999", "tables": ["fact_shipments"]}
]

# Save to metadata folder
output_path = os.path.join(metadata_folder, 'dictionary.json')

with open(output_path, 'w') as f:
    json.dump(red_flags_logic, f, indent=4)

print(f"✅ dictionary.json generated with 19 Red Flags at: {output_path}")

✅ dictionary.json generated with 19 Red Flags at: B:\3. Prog\2. Projects\7. Logistics and supply chain\metadata\dictionary.json


In [ ]:
# Checking whether all the 19 items are added

import json
import os

# Define the absolute path to your file
dict_path = r"B:\3. Prog\2. Projects\7. Logistics and supply chain\metadata\dictionary.json"

if os.path.exists(dict_path):
    with open(dict_path, 'r') as f:
        data = json.load(f)
    
    print(f"--- 📋 DICTIONARY INTEGRITY CHECK ---")
    print(f"Total Entries Found: {len(data)}")
    print("-" * 40)
    
    for i, entry in enumerate(data, 1):
        print(f"{i:02d}. {entry.get('name')}")
    
    print("-" * 40)
    if len(data) == 19:
        print("✅ SUCCESS: All 19 Red Flags are covered.")
    else:
        print(f"⚠️ WARNING: Found {len(data)} entries. Expected 19.")
else:
    print("❌ ERROR: dictionary.json not found at the specified path.")

--- 📋 DICTIONARY INTEGRITY CHECK ---
Total Entries Found: 19
----------------------------------------
01. RF01: Thermal Compliance Crisis
02. RF02: Supplier Integrity Audit
03. RF03: Status Casting Resolution
04. RF04: Thermal Volatility (RTE_01007)
05. RF05: Bimodal Peaks (RTE_00727)
06. RF06: Delay Frequency Ceiling
07. RF07: Supplier SUP_00726 Failure
08. RF08: Supplier SUP_00548 Failure
09. RF09: ID Padding Resolution
10. RF10: 39.25% FTR Ceiling
11. RF11: 47.64% Cost Variance
12. RF12: Statistical Outlier (Z-Score)
13. RF13: $28M Spend Concentration
14. RF14: Bottleneck RTE_00052
15. RF15: Mean Imputation Artifact
16. RF16: Geospatial Hot-Zones
17. RF17: Cost Volatility Audit
18. RF18: Temperature-Linked Delays
19. RF19: 0.1% Catastrophic Deviation
----------------------------------------
✅ SUCCESS: All 19 Red Flags are covered.


In [7]:
# This will include all the KPIs

import json
import os

# 1. Path Configuration
project_root = r"B:\3. Prog\2. Projects\7. Logistics and supply chain"
dict_path = os.path.join(project_root, "metadata", "dictionary.json")

# 2. Granular KPI Metrics (Extracted from Logistics_KPI_Analysis.ipynb)
granular_kpis = [
    # --- Operational Efficiency ---
    {"name": "KPI: Total Shipment Volume", "description": "Measures total network scale and throughput.", "sql": "SELECT COUNT(*) FROM fact_shipments", "tables": ["fact_shipments"]},
    {"name": "KPI: Fulfillment Status Distribution", "description": "Percentage breakdown of Delivered, Delayed, and Pending shipments.", "sql": "GROUP BY order_fulfillment_status", "tables": ["fact_shipments"]},
    {"name": "KPI: Volume by Shift", "description": "Identifies peak periods (Morning/Afternoon/Night) for staffing optimization.", "sql": "CASE WHEN HOUR(timestamp) BETWEEN 6 AND 14 THEN 'Morning'...", "tables": ["fact_shipments"]},
    {"name": "KPI: Asset Utilization Rate", "description": "Ratio of unique vehicles used in shipments vs. total fleet size.", "sql": "COUNT(DISTINCT vehicle_id) / Total Vehicles", "tables": ["fact_shipments", "dim_vehicles"]},
    {"name": "KPI: Average Delivery Time Deviation", "description": "Global measure of ETA accuracy (Positive = Late, Negative = Early).", "sql": "AVG(delivery_time_deviation)", "tables": ["fact_shipments"]},
    {"name": "KPI: Route Lead-Time Analysis", "description": "Compares delivery delays across different route risk levels.", "sql": "AVG(delivery_time_deviation) GROUP BY route_risk_level", "tables": ["fact_shipments", "dim_routes"]},
    {"name": "KPI: Bottleneck Identification", "description": "Impact Score = Volume * Average Delay for every route.", "sql": "COUNT(*) * AVG(delivery_time_deviation)", "tables": ["fact_shipments", "dim_routes"]},
    {"name": "KPI: Route Circuitry", "description": "Identifies planned vs. actual movement efficiency using GPS variance.", "sql": "ROUND(vehicle_gps_latitude, 1)", "tables": ["fact_shipments", "dim_routes"]},

    # --- Financial & Cost ---
    {"name": "KPI: Total Spend by Supplier", "description": "Total logistics financial throughput per vendor.", "sql": "SUM(shipping_costs) GROUP BY supplier_id", "tables": ["fact_shipments", "dim_suppliers"]},
    {"name": "KPI: Average Shipping Cost per Route", "description": "Geographical distribution of spend per corridor.", "sql": "AVG(shipping_costs) GROUP BY route_id", "tables": ["fact_shipments", "dim_routes"]},
    {"name": "KPI: Cost-to-Weight Efficiency", "description": "Measures transport cost per unit of weight (Inbound/Outbound).", "sql": "shipping_costs / weight_kg", "tables": ["fact_shipments"]},
    {"name": "KPI: Cost-to-Carrier Variance", "description": "Identifies carriers charging significantly above network averages.", "sql": "Actual Cost - Network Avg Cost", "tables": ["fact_shipments", "dim_suppliers"]},
    {"name": "KPI: Rolling Spend Trends", "description": "7-day moving average of total logistics costs.", "sql": "AVG(shipping_costs) OVER (ORDER BY timestamp ROWS BETWEEN 6 PRECEDING AND CURRENT ROW)", "tables": ["fact_shipments"]},

    # --- Quality & Compliance ---
    {"name": "KPI: Critical Temperature Breach Rate", "description": "Volume of product safety violations (>15°C).", "sql": "SUM(CASE WHEN iot_temperature > 15 THEN 1 ELSE 0 END) / COUNT(*)", "tables": ["fact_shipments"]},
    {"name": "KPI: Thermal Compliance Rate", "description": "% of shipments maintaining standard 0-15°C environment.", "sql": "COUNT(Compliant) / Total", "tables": ["fact_shipments"]},
    {"name": "KPI: Average Temperature Stability", "description": "Consistency of the cold chain using Standard Deviation.", "sql": "STDDEV(iot_temperature)", "tables": ["fact_shipments"]},
    {"name": "KPI: Condition-Based Risk Score", "description": "Composite score (0-100) based on Thermal, Delay, and Cost risks.", "sql": "Weighted Average (Risk Factors)", "tables": ["fact_shipments"]},

    # --- Supplier Performance ---
    {"name": "KPI: Load Distribution", "description": "Workload balance across the vendor network to identify over-reliance.", "sql": "COUNT(*) GROUP BY supplier_id", "tables": ["fact_shipments", "dim_suppliers"]},
    {"name": "KPI: Carrier Delay Frequency", "description": "% of late deliveries per supplier.", "sql": "Late_Count / Total_Count", "tables": ["fact_shipments", "dim_suppliers"]},
    {"name": "KPI: Vendor Thermal Integrity", "description": "Ranking suppliers by their ability to maintain compliance.", "sql": "AVG(iot_temperature) BY supplier_id", "tables": ["fact_shipments", "dim_suppliers"]},
    {"name": "KPI: Supplier Reliability Gap", "description": "Gap between promised reliability score and actual compliance.", "sql": "Actual % - Promised Reliability", "tables": ["dim_suppliers", "fact_shipments"]}
]

# 3. Execution: Load, Merge, and Save
if os.path.exists(dict_path):
    with open(dict_path, 'r') as f:
        existing_data = json.load(f)
    
    # Remove any broad 'KPI Categories' if they were added in the previous step
    # This keeps only the 19 Red Flags
    red_flags_only = [item for item in existing_data if not item['name'].startswith("KPI:")]
    
    # Combine 19 Red Flags + 21 Granular KPIs
    final_data = red_flags_only + granular_kpis
    
    with open(dict_path, 'w') as f:
        json.dump(final_data, f, indent=4)
    
    print(f"✅ Success: dictionary.json updated.")
    print(f"Total Entries: {len(final_data)} (19 Red Flags + 21 Granular KPIs)")
else:
    print("❌ Error: dictionary.json not found. Run the Red Flag script first.")

✅ Success: dictionary.json updated.
Total Entries: 40 (19 Red Flags + 21 Granular KPIs)


In [9]:
import json
import os
import pandas as pd
from IPython.display import display, HTML

# 1. Path Configuration
dict_path = r"B:\3. Prog\2. Projects\7. Logistics and supply chain\metadata\dictionary.json"

if os.path.exists(dict_path):
    # 2. Load the Dictionary
    with open(dict_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    # 3. Categorize into Lists
    rf_list = [
        {"ID": item['name'].split(":")[0], "Metric Name": item['name'], "Business Logic": item['description']} 
        for item in data if item['name'].startswith("RF")
    ]
    
    kpi_list = [
        {"ID": f"KPI_{i+1:02d}", "Metric Name": item['name'], "Measurement Goal": item['description']} 
        for i, item in enumerate([x for x in data if x['name'].startswith("KPI:")])
    ]

    # 4. Create DataFrames
    df_rf = pd.DataFrame(rf_list)
    df_kpi = pd.DataFrame(kpi_list)

    # 5. Visual Display
    display(HTML("<h2 style='color: #d9534f;'>🚩 Red Flag Intelligence (Domain Risks)</h2>"))
    display(df_rf.style.set_properties(**{'text-align': 'left'}).set_table_styles([dict(selector='th', props=[('text-align', 'left')])]))

    display(HTML("<h2 style='color: #5bc0de;'>📈 KPI Framework (Performance Metrics)</h2>"))
    display(df_kpi.style.set_properties(**{'text-align': 'left'}).set_table_styles([dict(selector='th', props=[('text-align', 'left')])]))

    # 6. Final Summary logic
    print("\n" + "="*60)
    print(f"✅ AUDIT COMPLETE: {len(rf_list)} Red Flags | {len(kpi_list)} KPIs | Total: {len(data)}")
    print("="*60)

else:
    print(f"❌ ERROR: dictionary.json not found at {dict_path}")

,ID,Metric Name,Business Logic
0,RF01,RF01: Thermal Compliance Crisis,"Global count of shipments where iot_temperature is < 0 or > 15 (11,598 breaches)."
1,RF02,RF02: Supplier Integrity Audit,Identification of carriers with frequent temperature violations above 8.0C.
2,RF03,RF03: Status Casting Resolution,Logic to handle order_fulfillment_status as VARCHAR to prevent numeric conversion errors.
3,RF04,RF04: Thermal Volatility (RTE_01007),Statistical variance audit to find inconsistent cooling on specific routes.
4,RF05,RF05: Bimodal Peaks (RTE_00727),Detection of two distinct temperature clusters indicating cooling equipment failure.
5,RF06,RF06: Delay Frequency Ceiling,Top 10 routes where delivery_time_deviation > 0 occurs in over 96% of cases.
6,RF07,RF07: Supplier SUP_00726 Failure,Documenting specific supplier with a near-total (>99%) delay rate.
7,RF08,RF08: Supplier SUP_00548 Failure,Documenting secondary supplier with a near-total (>99%) delay rate.
8,RF09,RF09: ID Padding Resolution,Handling leading zero inconsistencies for supplier IDs (e.g. SUP_00875 vs SUP_875).
9,RF10,RF10: 39.25% FTR Ceiling,Measuring the cap on 'Perfect Shipments' (On-Time + Thermal Integrity) for SUP_00358.


,ID,Metric Name,Measurement Goal
0,KPI_01,KPI: Total Shipment Volume,Measures total network scale and throughput.
1,KPI_02,KPI: Fulfillment Status Distribution,"Percentage breakdown of Delivered, Delayed, and Pending shipments."
2,KPI_03,KPI: Volume by Shift,Identifies peak periods (Morning/Afternoon/Night) for staffing optimization.
3,KPI_04,KPI: Asset Utilization Rate,Ratio of unique vehicles used in shipments vs. total fleet size.
4,KPI_05,KPI: Average Delivery Time Deviation,"Global measure of ETA accuracy (Positive = Late, Negative = Early)."
5,KPI_06,KPI: Route Lead-Time Analysis,Compares delivery delays across different route risk levels.
6,KPI_07,KPI: Bottleneck Identification,Impact Score = Volume * Average Delay for every route.
7,KPI_08,KPI: Route Circuitry,Identifies planned vs. actual movement efficiency using GPS variance.
8,KPI_09,KPI: Total Spend by Supplier,Total logistics financial throughput per vendor.
9,KPI_10,KPI: Average Shipping Cost per Route,Geographical distribution of spend per corridor.



✅ AUDIT COMPLETE: 19 Red Flags | 21 KPIs | Total: 40
